In [364]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import itertools

In [365]:
# Loading of three provided datasets
x_train = pd.read_csv("/content/x_train_id.csv")
y_train = pd.read_csv("/content/y_train_id.csv")

# Selecting of the columns containing the inputs
transf_cols = [col for col in x_train.columns if col != 'id']
transf_cols

['0', '1', '2', '3', '4', '5', '6']

In [367]:
# Function to normalise the inputs according to their corresponding sum (perimeter)
def perimeter_normalize_inputs(data):
    # Calculate the perimeter for each row using only the specified columns
    perimeters = data[transf_cols].sum(axis=1)
    # Create a copy of x_train to avoid modifying the original DataFrame
    x_train_normalized = data.copy()
    # Normalize specified columns by dividing each side length by the total perimeter
    x_train_normalized[transf_cols] = data[transf_cols].div(perimeters, axis=0)

    return x_train_normalized, perimeters.tolist()

# Function to normalise the outputs according to the sum of the inputs squared (perimeter squared)
def perimeter_normalize_outputs(perimeters, data):
    # Iterate over each row index and its corresponding perimeter
    for index, perimeter in enumerate(perimeters):
        # Divide the 'expected' value by the corresponding perimeter squared
        data.at[index, 'Expected'] /= (perimeter ** 2)
    return data

# Function to make the transform the outputs back to their expected form.
def reverse_perimeter_normalize_outputs(perimeters, data):
    # Iterate over each row index and its corresponding perimeter
    for index, perimeter in enumerate(perimeters):
        # Multiply the 'Expected' value by the square of the corresponding perimeter
        data.at[index, 'Expected'] *= (perimeter ** 2)
    return data


In [368]:
# Making of copies of the dataframes to ensure that the original dataframes are not altered
x_data = x_train.copy()
y_data = y_train.copy()

# Scaling and homogenisation of inputs and outputs using above methods
scaled_homogenized_inputs, train_perimeters= perimeter_normalize_inputs(x_data)
scaled_homogenized_outputs = perimeter_normalize_outputs(train_perimeters, y_data)
scaled_homogenized_inputs = scaled_homogenized_inputs.drop(columns=['id'])

,0,1,2,3,4,5,6
0,0.248949,0.078248,0.166808,0.108562,0.140465,0.010900,0.246067
1,0.064073,0.088627,0.083898,0.023048,0.335676,0.085884,0.318793
2,0.056001,0.296866,0.086534,0.328069,0.047537,0.163223,0.021770
3,0.239999,0.023813,0.251040,0.021952,0.139849,0.053286,0.270061
4,0.023304,0.016863,0.009576,0.271297,0.193578,0.112061,0.373322
...,...,...,...,...,...,...,...
9995,0.246619,0.078528,0.009851,0.023524,0.255800,0.267686,0.117992
9996,0.156113,0.076346,0.076927,0.209509,0.114732,0.006172,0.360200
9997,0.062604,0.111957,0.336439,0.013993,0.126326,0.075465,0.273215
9998,0.190516,0.030041,0.036243,0.192517,0.134106,0.027328,0.389249


In [369]:
# Function to compute the cartesian product of the inputs, increasing the dimensionality
def compute_multiplications(x_train):
    # Drop the 'id' column if it exists
    if 'id' in x_train.columns:
        x_train = x_train.drop(columns=['id'])

    # Get all combinations of 2 from column names
    combinations = list(itertools.combinations_with_replacement(x_train.columns, 2))

    # Initialize an empty list to store results
    rows_result = []

    # Compute multiplication for every row and each combination
    for _, row in x_train.iterrows():
        row_result = {}  # Dictionary to store multiplication results for this row
        for comb in combinations:
            col_name = f"{comb[0]}_{comb[1]}"  # New column name

            # Ensure that the combination is in the correct order
            if comb[0] <= comb[1]:
                row_result[col_name] = row[comb[0]] * row[comb[1]]
        rows_result.append(row_result)

    # Create DataFrame from the list of row results
    result_df = pd.DataFrame(rows_result)

    return result_df

# Calling of the above function
cartesian_inputs = compute_multiplications(scaled_homogenized_inputs)

In [372]:
# Get input columns
cart_transf_cols = [col for col in cartesian_inputs.columns if col != 'id']

# Get outputs in list form
outputs = scaled_homogenized_outputs['Expected'].tolist()

# Reshape to outputs to numpy array of (size, 1) instead of (size,), expected input for pytorch
np_outputs = np.array(outputs)
y_train_reshaped = np_outputs.reshape(-1, 1)

# Compile all inputs into a list of lists of integers
inputs = cartesian_inputs[cart_transf_cols].values.tolist()

array([[0.06871395],
       [0.05837871],
       [0.06030226],
       ...,
       [0.0613271 ],
       [0.05569457],
       [0.07094442]])

In [374]:
# Sort each inner list representing one input
inputs = [sorted(inner_list) for inner_list in inputs]

# Function to remove any duplicate inputs post sorting
def remove_duplicates(inputs):
    unique_tuples = set()
    result = []
    for inner_list in inputs:
        inner_tuple = tuple(inner_list)
        if inner_tuple not in unique_tuples:
            result.append(inner_list)
            unique_tuples.add(inner_tuple)
    return result

# Calling of function remove duplicate inputs
disambiguated_inputs = remove_duplicates(inputs)

[[0.0001188184525178033,
  0.0008529376284960481,
  0.0011833719297086069,
  0.0015311195466592487,
  0.001818267812292204,
  0.0026822270503075005,
  0.0027136453032972105,
  0.006122808222868046,
  0.008494829094018164,
  0.010991133509972328,
  0.011785788270659778,
  0.013052425805283346,
  0.015249179350094942,
  0.018109031418641574,
  0.019254352593377183,
  0.019479888355110123,
  0.019730328214893716,
  0.02343058110477244,
  0.026713630191096963,
  0.027026540163249103,
  0.027824784510827594,
  0.034563741390998555,
  0.0349686036009926,
  0.04104587298932241,
  0.04152666361501317,
  0.06054902918655082,
  0.06125826993373003,
  0.061975818368814675],
 [0.0005312326948177776,
  0.0014767916234922815,
  0.0019337297080894726,
  0.0019795065605510524,
  0.002042708115330428,
  0.004105382670328794,
  0.005375640209765835,
  0.0055028968205214895,
  0.005678592946908977,
  0.007038931565065752,
  0.0072055629874373545,
  0.007347697381973666,
  0.007376139046955224,
  0.007435

Min Max Scaler

In [375]:
def min_max_scaler(unscaled_data):
    # Flatten the list of lists
    flattened = [item for sublist in unscaled_data for item in sublist]

    # Compute the min and max values
    min_val = min(flattened)
    max_val = max(flattened)

    # Apply Min-Max scaling to each value
    scaled_flattened = [(item - min_val) / (max_val - min_val) for item in flattened]

    # Reshape back to the original list of lists structure
    scaled_data = []
    index = 0
    for sublist in unscaled_data:
        scaled_sublist = scaled_flattened[index:index+len(sublist)]
        scaled_data.append(scaled_sublist)
        index += len(sublist)

    return scaled_data


Standardization Scale

In [376]:
def standardize_data(unscaled_data):
    import numpy as np

    # Flatten the list of lists
    flattened = [item for sublist in unscaled_data for item in sublist]

    # Compute the mean and standard deviation
    mean_val = np.mean(flattened)
    std_val = np.std(flattened)

    # Apply standardization to each value
    standardized_flattened = [(item - mean_val) / std_val for item in flattened]

    # Reshape back to the original list of lists structure
    standardized_data = []
    index = 0
    for sublist in unscaled_data:
        standardized_sublist = standardized_flattened[index:index+len(sublist)]
        standardized_data.append(standardized_sublist)
        index += len(sublist)

    return standardized_data


Robust scaler

In [377]:
from sklearn.preprocessing import RobustScaler
import numpy as np

def robust_scaler(unscaled_data):
    # Flatten the list of lists
    flattened = [item for sublist in unscaled_data for item in sublist]

    # Reshape to a 2D array
    data_array = np.array(flattened).reshape(-1, 1)

    # Initialize the RobustScaler
    scaler = RobustScaler()

    # Fit the scaler to the data
    scaler.fit(data_array)

    # Transform the data using the fitted scaler
    scaled_data_array = scaler.transform(data_array)

    # Reshape back to the original list of lists structure
    scaled_data = []
    index = 0
    for sublist in unscaled_data:
        scaled_sublist = scaled_data_array[index:index+len(sublist)].flatten().tolist()
        scaled_data.append(scaled_sublist)
        index += len(sublist)

    return scaled_data, scaler

def robust_unscaler(scaled_data, scaler):
    # Flatten the scaled data
    scaled_flattened = [item for sublist in scaled_data for item in sublist]

    # Reshape to a 2D array
    scaled_array = np.array(scaled_flattened).reshape(-1, 1)

    # Use the inverse_transform method of the scaler to unscale the data
    unscaled_array = scaler.inverse_transform(scaled_array)

    # Reshape back to the original list of lists structure
    unscaled_data = []
    index = 0
    for sublist in scaled_data:
        unscaled_sublist = unscaled_array[index:index+len(sublist)].flatten().tolist()
        unscaled_data.append(unscaled_sublist)
        index += len(sublist)

    return unscaled_data


Standard Scaler

In [378]:
from sklearn.preprocessing import StandardScaler

def scale_with_standard_scaler(data):
    # Flatten the list of lists
    flattened = [item for sublist in data for item in sublist]

    # Reshape to a 2D array
    data_array = np.array(flattened).reshape(-1, 1)

    # Initialize the StandardScaler
    scaler = StandardScaler()

    # Fit the scaler to the data
    scaler.fit(data_array)

    # Transform the data using the fitted scaler
    scaled_data_array = scaler.transform(data_array)

    # Reshape back to the original list of lists structure
    scaled_data = []
    index = 0
    for sublist in data:
        scaled_sublist = scaled_data_array[index:index+len(sublist)].flatten().tolist()
        scaled_data.append(scaled_sublist)
        index += len(sublist)

    return scaled_data, scaler

def unscale_with_standard_scaler(scaled_data, scaler):
    # Flatten the list of lists
    flattened = [item for sublist in scaled_data for item in sublist]

    # Reshape to a 2D array
    scaled_array = np.array(flattened).reshape(-1, 1)

    # Use the inverse_transform method of the scaler to unscale the data
    unscaled_array = scaler.inverse_transform(scaled_array)

    # Reshape back to the original list of lists structure
    unscaled_data = []
    index = 0
    for sublist in scaled_data:
        unscaled_sublist = unscaled_array[index:index+len(sublist)].flatten().tolist()
        unscaled_data.append(unscaled_sublist)
        index += len(sublist)

    return unscaled_data


In [379]:
scaled_disambiguated_inputs = np.array(disambiguated_inputs, dtype=np.float32)
scaled_outputs = np.array(y_train_reshaped, dtype=np.float32)

array([[0.06871395],
       [0.05837871],
       [0.06030226],
       ...,
       [0.0613271 ],
       [0.05569457],
       [0.07094442]], dtype=float32)

In [381]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from math import floor
from sklearn.model_selection import train_test_split
from torch.optim.lr_scheduler import StepLR

In [382]:
print(type(scaled_disambiguated_inputs[0][0]), type(scaled_outputs[0][0]))

<class 'numpy.float32'> <class 'numpy.float32'>


Defined Variables

In [383]:
datatype=torch.float32

# Split data into train and validation data respectively
x_train_data, x_test_data, y_train_data, y_test_data = train_test_split(scaled_disambiguated_inputs,
            scaled_outputs, test_size=0.2, random_state=42)

# Ensure that the elements are lists of float32
x_train_data = [list(map(np.float32, sublist)) for sublist in x_train_data]
x_test_data = [list(map(np.float32, sublist)) for sublist in x_test_data]
y_train_data = [list(map(np.float32, sublist)) for sublist in y_train_data]
y_test_data = [list(map(np.float32, sublist)) for sublist in y_test_data]

print(len(x_test_data))

2000


In [384]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super(NeuralNetwork, self).__init__()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28, 256, bias=False),
            nn.SELU(0.1),
            nn.Linear(256, 128, bias=False),
            nn.SELU(0.1),
            nn.Linear(128, 64, bias=False),
            nn.SELU(0.1),
            nn.Linear(64, 32, bias=False),
            nn.SELU(0.1),
            nn.Linear(32, 1, bias=False),
        )

    def forward(self, x):
        logits = self.linear_relu_stack(x)
        return logits

In [385]:
class NumpyDataset(Dataset):
    def __init__(self, x, y, dtype=datatype):
        self.x = torch.tensor(x, dtype=dtype)
        self.y = torch.tensor(y, dtype=dtype)

    def __repr__(self):
        return f"NumpyDataset(data={self.x, self.y})"

    def __len__(self):
        return self.x.size()[0]

    def __getitem__(self, index):
        return self.x[index], self.y[index]

In [386]:
#Method converting tabel data to tensors
def np_to_tensor(x, dtype=datatype):
    return torch.tensor(x, dtype=dtype)

#Uses above method to set up tensor data
def prepare_data(x_train, y_train, x_test, y_test):
    data_dict = {}
    data_dict["train"] = NumpyDataset(x_train, y_train)
    data_dict["x_test"] = np_to_tensor(x_test)
    data_dict["y_test"] = np_to_tensor(y_test)
    return data_dict

# Callling of the functions
training_data = prepare_data(x_train_data, y_train_data, x_test_data, y_test_data)

In [388]:
#Set up cuda for GPU Maxing
import torch
#for catching best copy
import copy

# Check if CUDA is available
if torch.cuda.is_available():
    devicet = torch.device("cuda")
    print("CUDA is available. Using GPU.")
else:
    devicet = torch.device("cpu")
    print("CUDA is not available. Using CPU.")

# Print device name
print(f"Using device: {devicet}")

CUDA is available. Using GPU.
Using device: cuda


In [389]:
def train_model(
    data, model, nepochs=1, lr=1e-6, batch_size=1, print_every=1, loss_fn=nn.MSELoss()
):
    torch.manual_seed(0)

    train_data = data["train"]
    x_test = data["x_test"]
    y_test = data["y_test"]

    # Move the model to the GPU
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    loss_fn = loss_fn.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_dict = {"train": [], "test": []}
    num_batches = floor(len(train_data) / batch_size)

    # Initialize variables to keep track of the best model and the lowest test loss
    best_model = None
    lowest_test_loss = float('inf')


    for epoch in tqdm(range(nepochs)):

        epoch_loss_sum = 0

        for x_batch, y_batch in DataLoader(
            train_data, batch_size=batch_size, shuffle=True
        ):
            # Move batch to the GPU
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)

            y_pred = model(x_batch)
            loss = loss_fn(y_pred, y_batch)
            epoch_loss_sum += loss.item()
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        loss_dict["train"].append(epoch_loss_sum / num_batches)

        with torch.no_grad():
            # Move test data to the GPU
            x_test, y_test = x_test.to(device), y_test.to(device)

            y_pred = model(x_test)
            loss = loss_fn(y_pred, y_test)
            loss_dict["test"].append(loss.item())

            # Check if this is the lowest test loss we've encountered
            if loss.item() < lowest_test_loss:
                lowest_test_loss = loss.item()
                # Make a deep copy of the model to store the best model
                best_model = copy.deepcopy(model)
                print(loss.item())

            if (epoch + 1) % print_every == 0:
                print(
                    #add to print best model as well
                    epoch + 1, f"Train: {loss_dict['train'][-1]}  Test: {loss.item()}"
                )

        # Ask the user whether to continue every 5000 epochs
        if (epoch + 1) % 5000 == 0:
            user_input = input("Do you want to continue training? (y/n): ")
            if user_input.lower() != 'y':
                print("Training stopped by user.")
                break

    # Move model and data back to CPU
    best_model = best_model.to("cpu")
    model = model.to("cpu")
    data["x_test"] = x_test.to("cpu")
    data["y_test"] = y_test.to("cpu")

    # Create a new list for train_data with all tensors moved to the CPU
    new_train_data = [(x.to("cpu"), y.to("cpu")) for x, y in train_data]
    data["train"] = new_train_data

    return model, best_model, loss_dict

In [390]:
class MAPELoss(torch.nn.Module):
    def __init__(self):
        super(MAPELoss, self).__init__()

    def forward(self, y_pred, y_true):
        """
        Calculates the Mean Absolute Percentage Error between y_pred and y_true.

        Args:
            y_pred (torch.Tensor): Predicted values.
            y_true (torch.Tensor): True values.

        Returns:
            torch.Tensor: Mean Absolute Percentage Error.
        """
        epsilon = 1e-8  # to avoid division by zero
        percentage_error = torch.abs((y_true - y_pred) / (y_true + epsilon))
        mean_percentage_error = torch.mean(percentage_error)*100
        return mean_percentage_error

In [391]:
# Define model arguments
total_epochs = 100000
print_every = 100
batch_size = 56
lr = 0.00001
loss_fn = MAPELoss()

In [ ]:
model = NeuralNetwork()

model, best_model, loss_dict = train_model(
    training_data,
    model,
    nepochs=total_epochs,
    batch_size=batch_size,
    lr=lr,
    print_every=print_every,
    loss_fn=loss_fn,
)

  0%|          | 1/100000 [00:00<9:23:36,  2.96it/s]

25.00054931640625


  0%|          | 2/100000 [00:00<9:04:41,  3.06it/s]

12.73844051361084


  0%|          | 3/100000 [00:00<8:49:10,  3.15it/s]

...

 21%|██        | 20600/100000 [2:18:07<6:32:21,  3.37it/s]

20600 Train: 0.03184940785327008  Test: 0.03270793706178665


 21%|██        | 20700/100000 [2:18:39<6:39:32,  3.31it/s]

20700 Train: 0.0381982772755371  Test: 0.03498747944831848


 21%|██        | 20722/100000 [2:18:46<7:19:32,  3.01it/s]

In [ ]:
def plot_loss(loss_dict):
    plt.plot(loss_dict["train"], label="train")
    plt.plot(loss_dict["test"], label="test")
    plt.yscale("log")
    plt.legend()
    plt.show()

In [ ]:
plot_loss(loss_dict)

In [ ]:
def plot_result(data_dict, model):
    with torch.no_grad():
        y_pred = model(data_dict["x_test"])
        loss = loss_fn(y_pred, data_dict["y_test"])
        print(loss.item())
        plt.plot(y_pred.cpu().numpy(), data_dict["y_test"].cpu().numpy(), "o")
        # plot line x = y
        x = [0.02, 0.09]
        plt.plot(x, x, "r")
        plt.show()

plot_result(training_data, model)
plot_result(training_data, best_model)

In [ ]:
x_kaggle_test = pd.read_csv("x_test_id.csv")
x_kaggle_test

In [ ]:
scaled_homogenized_kaggle_inputs, kaggle_perimeters= perimeter_normalize_inputs(x_kaggle_test)
scaled_homogenized_kaggle_inputs = compute_multiplications(scaled_homogenized_kaggle_inputs)

# Compiling all inputs into a list of lists of integers
kaggle_inputs = scaled_homogenized_kaggle_inputs[cart_transf_cols].values.tolist()
# Sort each inner list
sorted_kaggle_inputs = [sorted(inner_list) for inner_list in kaggle_inputs]
# Calling of function remove duplicate inputs
disambiguated_sorted_kaggle_inputs = remove_duplicates(sorted_kaggle_inputs)
# final_kaggle_inputs = min_max_scaler(disambiguated_sorted_kaggle_inputs)


tensor_kaggle_input = np_to_tensor(disambiguated_sorted_kaggle_inputs)
scaled_homogenized_kaggle_inputs

In [ ]:
kaggle_predictions = model(tensor_kaggle_input)
kaggle_predictions

In [ ]:
# Detach the tensor and convert it to a NumPy array
flattened_predictions = kaggle_predictions.detach().numpy().flatten()

# Create a DataFrame with Id and prediction
df = pd.DataFrame({
    'id': range(len(flattened_predictions)),
    'Expected': flattened_predictions
})

df


In [ ]:
predictions = reverse_perimeter_normalize_outputs(kaggle_perimeters, df)

# Save the DataFrame to a CSV file
csv_file_path = 'final_bestsofar_predictions.csv'
predictions.to_csv(csv_file_path, index=False)

# Display the DataFrame
predictions